```
Incoming Data
     ↓
Pydantic Model
     ↓
Check:
- required fields?
- correct type?
- valid range?
- correct format?
     ↓
Valid → continue
Invalid → error
```

##### Pydantic is a Python library used to define the expected structure of data and validate incoming data against that structure.

##### Pydantic helps us define what our data should look like and checks whether the received data follows that structure.

##### Pydantic is a Python library that lets us define the expected structure of data using Python types and automatically validates the input data against that structure.

##### Schema enforcement simply means making sure that the data follows the structure and rules we have already defined.

##### For example, if our schema says name should be a string and age should be an integer, then the incoming data should follow these rules. If it doesn’t, Pydantic will raise a validation error.

```
Schema defining = Defining what the data structure should look like.

Schema validation = Checking whether the input data follows the defined structure and rules.

Schema enforcement = Ensuring that invalid data is not accepted and the defined structure and rules are followed.
```

```
Raw / Untrusted Data
        ↓
   Pydantic Schema
        ↓
 ┌───────────────────┐
 │ Type checking     │
 │ Type conversion   │
 │ Constraints       │
 │ Custom validation │
 └───────────────────┘
        ↓
 Valid Python Object
 ```

What is BaseModel?

You will see this everywhere:

from pydantic import BaseModel

Then:

class User(BaseModel):
    name: str
    age: int

BaseModel gives your class Pydantic capabilities:

Validation
Parsing
Serialization
JSON Schema
Error messages

Without BaseModel:

class User:
    ...

it's just a normal Python class.

!pip install pydantic
!pip install pydantic-settings

Pydantic V1
      ↓
major API changes
      ↓
Pydantic V2

dict()              → model_dump()
json()              → model_dump_json()
parse_obj()         → model_validate()
parse_raw()         → model_validate_json()
schema()            → model_json_schema()
copy()              → model_copy()

@validator           → @field_validator
@root_validator      → @model_validator

What is Field()?

Example:

age: int = Field(
    ge=18,
    description="Age of the user"
)

Field() lets you add:

Validation rules
Description
Default values
Constraints
Metadata

Useful constraints:

Field(gt=0)

greater than.

Field(ge=0)

greater than or equal.

Field(lt=100)

less than.

Field(le=100)

less than or equal.

For strings:

Field(
    min_length=3,
    max_length=50
)

In [ ]:
from pydantic import ValidationError

try:

    user = UserRegistration(
        name="Sunny",
        email="wrong",
        age=10
    )

except ValidationError as e:

    print(e)

Python class
+
type annotations
+
validation
+
serialization
+
JSON Schema

Pydantic tells you exactly:

email
→ invalid email

age
→ must be >= 18

This is much better than manually writing dozens of if conditions.

In [ ]:
#Without Pydantic vs with Pydantic

def create_user(data):

    if "name" not in data:
        raise Exception(...)

    if not isinstance(
        data["age"],
        int
    ):
        raise Exception(...)

    if data["age"] < 18:
        raise Exception(...)

    ...

In [ ]:
#With Pydantic:

class User(BaseModel):
    name: str
    age: int = Field(ge=18)

In [1]:
from pydantic import BaseModel

In [2]:
class User(BaseModel):
    name: str
    age: int

In [3]:
user = User(
    name="Sunny",
    age=30
)

In [4]:
user

User(name='Sunny', age=30)

In [5]:
user.name

'Sunny'

In [6]:
user.age

30

In [7]:
user = User(
    name="Sunny",
    age="abc"
)

ValidationError: 1 validation error for User
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='abc', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing

##### fail because age should be an integer.

1. User Registration

Signup Form
 ↓
Backend API
 ↓
Validate user data

In [ ]:
from pydantic import BaseModel, EmailStr, Field

class UserRegistration(BaseModel):
    name: str
    email: EmailStr
    age: int = Field(ge=18)

In [ ]:
user = UserRegistration(
    name="Sunny",
    email="sunny@gmail.com",
    age=30
)

print(user)

In [ ]:
#Invalid:
UserRegistration(
    name="Sunny",
    email="wrong-email",
    age=15
)

Pydantic rejects it.
Pydantic = API input validation

2. Product / E-commerce Validation

In [ ]:
from pydantic import BaseModel, Field

class Product(BaseModel):
    name: str
    price: float = Field(gt=0)
    quantity: int = Field(ge=0)

In [ ]:
product = Product(
    name="Laptop",
    price=75000,
    quantity=5
)

In [ ]:
Product(
    name="Laptop",
    price=-500,
    quantity=-2
)

Pydantic automatically protects your application from invalid business input.

Concept
Field(gt=0)

means:

value must be greater than zero.

Field(ge=0)

means:

value must be greater than or equal to zero.

So:

Pydantic
=
type validation
+
business constraints

3. Banking Transaction

In [ ]:
from pydantic import BaseModel, Field

class MoneyTransfer(BaseModel):
    sender_account: str
    receiver_account: str
    amount: float = Field(gt=0)

In [ ]:
transfer = MoneyTransfer(
    sender_account="ACC001",
    receiver_account="ACC002",
    amount=5000
)

In [ ]:
MoneyTransfer(
    sender_account="ACC001",
    receiver_account="ACC002",
    amount=-5000
)

Why?

Because:

amount = -5000

does not make sense for a transfer request.

Important point

Pydantic validates the shape/input.

It does NOT automatically check:

Does ACC001 actually exist?
Does user have ₹5000?
Is account blocked?

Those are business/database checks.

So:

Pydantic
=
input validation

Service layer
=
business validation

Configuration / Environment Settings

Pydantic is also useful for application configuration.

Suppose your GenAI application needs:

Model name
Temperature
API URL
Timeout

In [ ]:
from pydantic import BaseModel, Field

class ModelConfig(BaseModel):
    model_name: str
    temperature: float = Field(ge=0, le=2)
    timeout: int = Field(gt=0)

In [ ]:
config = ModelConfig(
    model_name="gpt-model",
    temperature=0.7,
    timeout=30
)

In [ ]:
ModelConfig(
    model_name="gpt-model",
    temperature=10,
    timeout=-5
)

Pydantic can protect configuration from bad values too.

Configuration
 ↓
Pydantic
 ↓
Safe application startup

LLM Structured Output / Tool Calling

Suppose you want the LLM to return:

name
age
city

Instead of random text:

"Sunny is around 30 and lives in Bangalore..."

In [ ]:
from pydantic import BaseModel, Field

class PersonDetails(BaseModel):
    name: str
    age: int
    city: str

In [ ]:
structured_model = model.with_structured_output(
    PersonDetails
)

In [ ]:
PersonDetails(
    name="Sunny",
    age=30,
    city="Bangalore"
)

This is extremely useful for:

Tool calling
Agents
API responses
Extraction
Classification
Routing
RAG pipelines

In [ ]:
from typing import Literal

class RouteDecision(BaseModel):
    route: Literal[
        "RAG",
        "WEB",
        "LLM"
    ]

    reasoning: str

Then the model cannot randomly return:

"Maybe search something."

You expect:

{
    "route": "RAG",
    "reasoning": "Question requires internal documents."
}

Concept
LLM unstructured text
        ↓
Pydantic schema
        ↓
Structured predictable output

In [ ]:
# Required fields vs optional fields vs defaults

# Students must understand these carefully.

# Required

class User(BaseModel):
    name: str

In [ ]:
class User(BaseModel):
    name: str

In [ ]:
class User(BaseModel):
    country: str = "India"

In [ ]:
class User(BaseModel):
    middle_name: str | None
    
# This means:

# Field is required
# BUT
# value may be None

# It does not automatically mean optional-to-provide.

In [ ]:
# ============================================================
# 1. IMPORTS
# ============================================================

from typing import Literal
from typing_extensions import TypedDict

from pydantic import BaseModel, Field

from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

from langgraph.graph import (
    StateGraph,
    START,
    END,
)


# ============================================================
# 2. LLM
# ============================================================

model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)


# ============================================================
# 3. PYDANTIC MODEL #1
#    STRUCTURED INTENT DECISION
# ============================================================

class IntentDecision(BaseModel):

    intent: Literal[
        "refund",
        "order_status",
        "general"
    ] = Field(
        description="Type of customer request"
    )

    order_id: str | None = Field(
        default=None,
        description="Order ID if mentioned by customer"
    )

    reason: str = Field(
        description="Reason for selecting this intent"
    )


# Make LLM return this structure
intent_model = model.with_structured_output(
    IntentDecision
)


# ============================================================
# 4. PYDANTIC MODEL #2
#    TOOL INPUT VALIDATION
# ============================================================

class OrderInput(BaseModel):

    order_id: str = Field(
        min_length=3,
        description="Customer order ID"
    )


class RefundInput(BaseModel):

    order_id: str = Field(
        min_length=3,
        description="Order ID to refund"
    )

    reason: str = Field(
        min_length=5,
        description="Reason for requesting refund"
    )


# ============================================================
# 5. TOOLS
# ============================================================

@tool(args_schema=OrderInput)
def get_order_details(
    order_id: str
) -> dict:
    """
    Get order information from the order database.
    """

    # Fake database for demo
    database = {

        "ORD101": {
            "product": "Laptop",
            "amount": 75000,
            "status": "Delivered",
            "refundable": True
        },

        "ORD102": {
            "product": "Headphones",
            "amount": 5000,
            "status": "Shipped",
            "refundable": False
        }
    }

    return database.get(
        order_id,
        {
            "error": "Order not found"
        }
    )


@tool(args_schema=RefundInput)
def create_refund(
    order_id: str,
    reason: str
) -> dict:
    """
    Create a refund request for an eligible order.
    """

    return {
        "refund_id": "REF-9001",
        "order_id": order_id,
        "status": "Refund initiated",
        "reason": reason
    }


# ============================================================
# 6. PYDANTIC MODEL #3
#    FINAL RESPONSE
# ============================================================

class CustomerResponse(BaseModel):

    success: bool

    message: str

    order_id: str | None = None

    reference_id: str | None = None


final_model = model.with_structured_output(
    CustomerResponse
)


# ============================================================
# 7. LANGGRAPH STATE
# ============================================================

class SupportState(TypedDict, total=False):

    question: str

    intent: str

    order_id: str

    reason: str

    order_details: dict

    refund_details: dict

    final_response: CustomerResponse


# ============================================================
# 8. INTENT CLASSIFIER NODE
# ============================================================

def classify_intent(
    state: SupportState
):

    decision = intent_model.invoke(
        f"""
        Analyze the following customer request.

        Customer:
        {state["question"]}

        Identify whether the request is:

        refund,
        order_status,
        or general.
        """
    )

    print("\nINTENT DECISION")
    print(decision)

    return {

        "intent":
            decision.intent,

        "order_id":
            decision.order_id,

        "reason":
            decision.reason
    }


# ============================================================
# 9. ROUTER
# ============================================================

def route_intent(
    state: SupportState
):

    return state["intent"]


# ============================================================
# 10. REFUND NODE
# ============================================================

def refund_node(
    state: SupportState
):

    print("\nGETTING ORDER DETAILS...")

    order = get_order_details.invoke({
        "order_id":
            state["order_id"]
    })


    print(
        "Order:",
        order
    )


    # Order does not exist
    if "error" in order:

        return {
            "order_details": order,

            "refund_details": {
                "error":
                    "Refund cannot be created."
            }
        }


    # Order exists but cannot be refunded
    if not order["refundable"]:

        return {

            "order_details": order,

            "refund_details": {
                "error":
                    "Order is not eligible for refund."
            }
        }


    print(
        "\nCREATING REFUND..."
    )


    refund = create_refund.invoke({

        "order_id":
            state["order_id"],

        "reason":
            state["question"]
    })


    print(
        "Refund:",
        refund
    )


    return {

        "order_details":
            order,

        "refund_details":
            refund
    }


# ============================================================
# 11. ORDER STATUS NODE
# ============================================================

def order_status_node(
    state: SupportState
):

    order = get_order_details.invoke({

        "order_id":
            state["order_id"]
    })


    return {
        "order_details":
            order
    }


# ============================================================
# 12. GENERAL NODE
# ============================================================

def general_node(
    state: SupportState
):

    return {

        "order_details": {},

        "refund_details": {}
    }


# ============================================================
# 13. FINAL RESPONSE NODE
# ============================================================

def final_response_node(
    state: SupportState
):

    response = final_model.invoke(
        f"""
        Create a professional customer response.

        Original request:
        {state["question"]}

        Intent:
        {state.get("intent")}

        Order details:
        {state.get("order_details")}

        Refund details:
        {state.get("refund_details")}

        Return a structured customer response.
        """
    )


    return {
        "final_response":
            response
    }


# ============================================================
# 14. BUILD LANGGRAPH
# ============================================================

workflow = StateGraph(
    SupportState
)


workflow.add_node(
    "classifier",
    classify_intent
)


workflow.add_node(
    "refund",
    refund_node
)


workflow.add_node(
    "order_status",
    order_status_node
)


workflow.add_node(
    "general",
    general_node
)


workflow.add_node(
    "final",
    final_response_node
)


# ============================================================
# 15. EDGES
# ============================================================

workflow.add_edge(
    START,
    "classifier"
)


workflow.add_conditional_edges(

    "classifier",

    route_intent,

    {
        "refund":
            "refund",

        "order_status":
            "order_status",

        "general":
            "general"
    }
)


workflow.add_edge(
    "refund",
    "final"
)


workflow.add_edge(
    "order_status",
    "final"
)


workflow.add_edge(
    "general",
    "final"
)


workflow.add_edge(
    "final",
    END
)


# ============================================================
# 16. COMPILE
# ============================================================

app = workflow.compile()


# ============================================================
# 17. VISUALIZE
# ============================================================

from IPython.display import (
    Image,
    display
)


display(
    Image(
        app.get_graph().draw_mermaid_png()
    )
)


# ============================================================
# 18. TEST
# ============================================================

result = app.invoke({

    "question":
        "My order ORD101 arrived damaged. "
        "Please refund it."
})


# ============================================================
# 19. FINAL RESULT
# ============================================================

print("\nFINAL RESPONSE")

print(
    result["final_response"]
)

What exactly is Pydantic doing?
Pydantic #1 — LLM decision

Instead of LLM saying:

"Looks like the customer probably wants a refund."

we force:

IntentDecision(
    intent="refund",
    order_id="ORD101",
    reason="Customer requested refund for damaged item."
)

So:

LLM
 ↓
Pydantic
 ↓
Predictable Decision
Pydantic #2 — Tool safety

The refund tool expects:

class RefundInput(BaseModel):
    order_id: str
    reason: str

If something invalid comes:

create_refund.invoke({
    "order_id": "",
    "reason": "bad"
})

Pydantic rejects it before your actual function runs.

So:

LLM-generated arguments
        ↓
Pydantic validation
        ↓
Valid?
 /        \
Yes        No
 ↓          ↓
Tool       Error

This is especially important for real tools like:

Transfer Money
Create Customer
Book Flight
Update Database
Send Email
Pydantic #3 — final agent response

Instead of getting unpredictable:

"Okay yeah refund probably initiated..."

you get:

CustomerResponse(
    success=True,
    message="Your refund has been initiated.",
    order_id="ORD101",
    reference_id="REF-9001"
)

Now your frontend/API can easily consume it:

result["final_response"].success
result["final_response"].message
result["final_response"].reference_id